# 🤲 InterpreteSenales — Entrenamiento en Google Colab

Este notebook entrena el modelo LSTM usando GPU de Colab.
Los archivos `.h5` de keypoints deben estar ya subidos a Google Drive.

## Pasos antes de ejecutar:
1. En Google Drive, crea la carpeta `InterpreteSenales_Training/keypoints/`
2. Sube todos los archivos `.h5` de `data/keypoints/` local a esa carpeta
3. En Colab: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**
4. Ejecuta las celdas en orden

## Celda 1 — Verificar GPU disponible

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU disponible: {gpus[0].name}')
    print(f'   TensorFlow version: {tf.__version__}')
else:
    print('⚠️  No se detectó GPU.')
    print('   Ve a: Entorno de ejecución → Cambiar tipo de entorno → GPU')

## Celda 2 — Instalar dependencias

In [ ]:
# TensorFlow ya viene en Colab. Solo faltan tables y scikit-learn
!pip install tables scikit-learn --quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.regularizers import l2
import os, shutil, warnings
warnings.filterwarnings('ignore')
print('✅ Dependencias listas')

## Celda 3 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── CONFIGURA AQUÍ la ruta de tu carpeta en Drive ──────────────────
DRIVE_KEYPOINTS = '/content/drive/MyDrive/InterpreteSenales_Training/keypoints'
DRIVE_OUTPUT    = '/content/drive/MyDrive/InterpreteSenales_Training/output'
# ───────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_OUTPUT, exist_ok=True)

if os.path.exists(DRIVE_KEYPOINTS):
    h5_files = [f for f in os.listdir(DRIVE_KEYPOINTS) if f.endswith('.h5')]
    print(f'✅ Drive montado. Archivos .h5 encontrados: {len(h5_files)}')
    for f in sorted(h5_files):
        size = os.path.getsize(os.path.join(DRIVE_KEYPOINTS, f)) / 1e6
        print(f'   • {f:<25} {size:.1f} MB')
else:
    print(f'❌ Carpeta no encontrada: {DRIVE_KEYPOINTS}')
    print('   Crea la carpeta en Drive y sube los archivos .h5')

## Celda 4 — Copiar .h5 a Colab (más rápido que leer desde Drive)

In [ ]:
LOCAL_KEYPOINTS = '/content/keypoints'
os.makedirs(LOCAL_KEYPOINTS, exist_ok=True)

print('Copiando archivos .h5 a Colab...')
for f in sorted(os.listdir(DRIVE_KEYPOINTS)):
    if f.endswith('.h5'):
        src = os.path.join(DRIVE_KEYPOINTS, f)
        dst = os.path.join(LOCAL_KEYPOINTS, f)
        shutil.copy2(src, dst)
        print(f'   ✅ {f}')

print(f'\nListo. Keypoints en: {LOCAL_KEYPOINTS}')

## Celda 5 — Configuración y arquitectura del modelo

In [ ]:
# Constantes — deben coincidir con constants.py local
MODEL_FRAMES     = 15
LENGTH_KEYPOINTS = 1662
KEYPOINTS_PATH   = LOCAL_KEYPOINTS
MODEL_PATH       = '/content/actions_15.keras'

# ── Arquitectura LSTM (igual a model.py local) ──────────────────────
def get_model(max_length_frames, output_length):
    model = Sequential([
        LSTM(64, return_sequences=True,
             input_shape=(max_length_frames, LENGTH_KEYPOINTS),
             kernel_regularizer=l2(0.01)),
        Dropout(0.5),
        LSTM(128, return_sequences=False, kernel_regularizer=l2(0.001)),
        Dropout(0.5),
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        Dense(output_length, activation='softmax'),
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

print(f'✅ Configuración lista — MODEL_FRAMES={MODEL_FRAMES}, LENGTH_KEYPOINTS={LENGTH_KEYPOINTS}')

## Celda 6 — Cargar datos desde los .h5

In [ ]:
def get_gestures_with_valid_keypoints():
    valid = []
    for f in sorted(os.listdir(KEYPOINTS_PATH)):
        if not f.endswith('.h5'):
            continue
        gesture = f[:-3]
        try:
            data = pd.read_hdf(os.path.join(KEYPOINTS_PATH, f), key='data')
            if not data.empty:
                valid.append(gesture)
                print(f'   ✅ {gesture:<20} {len(data.groupby("sample"))} muestras')
        except Exception as e:
            print(f'   ❌ {gesture}: {e}')
    return valid

def get_sequences_and_labels(word_ids):
    sequences, labels = [], []
    for idx, gesture in enumerate(word_ids):
        path = os.path.join(KEYPOINTS_PATH, f'{gesture}.h5')
        data = pd.read_hdf(path, key='data')
        for _, sample in data.groupby('sample'):
            seq = [row['keypoints'] for _, row in sample.iterrows()]
            sequences.append(seq)
            labels.append(idx)
    return sequences, labels

print('Gestos detectados:')
word_ids = get_gestures_with_valid_keypoints()
print(f'\nTotal: {len(word_ids)} gestos → {word_ids}')

if len(word_ids) < 2:
    raise ValueError('Se necesitan al menos 2 gestos con keypoints válidos.')

## Celda 7 — Preparar datos para entrenamiento

In [ ]:
print('Cargando secuencias...')
sequences, labels = get_sequences_and_labels(word_ids)

sequences = pad_sequences(sequences, maxlen=MODEL_FRAMES,
                          padding='pre', truncating='post', dtype='float32')
X = np.array(sequences)
y = to_categorical(labels).astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)

print(f'✅ Dataset listo:')
print(f'   X shape:      {X.shape}  →  (muestras, frames, keypoints)')
print(f'   Train:        {X_train.shape[0]} muestras')
print(f'   Validación:   {X_val.shape[0]} muestras')
print(f'   Clases:       {len(word_ids)} gestos')

# Verificar balance de clases
from collections import Counter
counts = Counter(labels)
print('\nMuestras por gesto:')
for i, g in enumerate(word_ids):
    bal = '✅' if counts[i] >= 100 else '⚠️ pocas muestras'
    print(f'   {g:<20} {counts[i]:>4} muestras  {bal}')

## Celda 8 — Entrenar el modelo
**⏱️ Con GPU T4: ~3-5 min para 500 épocas con 5 gestos**

In [ ]:
# ── Configura épocas aquí ──────────────────────────────────────────
EPOCHS = 500
# ───────────────────────────────────────────────────────────────────

model = get_model(MODEL_FRAMES, len(word_ids))
model.summary()

print(f'\n🚀 Iniciando entrenamiento — {EPOCHS} épocas...')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=16,
    verbose=1
)

final_acc     = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
print(f'\n✅ Entrenamiento completado')
print(f'   Accuracy final:     {final_acc:.4f} ({final_acc*100:.1f}%)')
print(f'   Val accuracy final: {final_val_acc:.4f} ({final_val_acc*100:.1f}%)')

## Celda 9 — Gráficas de entrenamiento

In [ ]:
acc     = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss    = history.history['loss']
val_loss= history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc,     label='Entrenamiento', linewidth=2)
plt.plot(epochs_range, val_acc, label='Validación',    linewidth=2)
plt.legend()
plt.title('Precisión', fontsize=14, fontweight='bold')
plt.xlabel('Época'); plt.ylabel('Precisión'); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss,     label='Entrenamiento', linewidth=2)
plt.plot(epochs_range, val_loss, label='Validación',    linewidth=2)
plt.legend()
plt.title('Pérdida', fontsize=14, fontweight='bold')
plt.xlabel('Época'); plt.ylabel('Pérdida'); plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfica guardada en /content/training_history.png')

## Celda 10 — Matriz de confusión

In [ ]:
y_pred = np.argmax(model.predict(X, verbose=0), axis=1)
y_true = np.array(labels)
cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(word_ids)))

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap=plt.cm.Blues)
ax.set(xticks=np.arange(len(word_ids)), yticks=np.arange(len(word_ids)),
       xticklabels=word_ids, yticklabels=word_ids,
       title='Matriz de Confusión', ylabel='Real', xlabel='Predicho')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
thresh = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontweight='bold',
                color='white' if cm[i, j] > thresh else 'black')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

acc_por_gesto = cm.diagonal() / cm.sum(axis=1) * 100
print('\nAccuracy por gesto:')
for g, a in zip(word_ids, acc_por_gesto):
    icono = '✅' if a >= 90 else ('⚠️' if a >= 60 else '❌')
    print(f'   {icono} {g:<20} {a:.1f}%')

## Celda 11 — Guardar modelo en Google Drive

In [ ]:
# Guardar modelo localmente en Colab
model.save(MODEL_PATH)
print(f'✅ Modelo guardado en Colab: {MODEL_PATH}')

# Copiar a Google Drive
drive_model_path = os.path.join(DRIVE_OUTPUT, 'actions_15.keras')
shutil.copy2(MODEL_PATH, drive_model_path)

# Copiar gráficas también
shutil.copy2('/content/training_history.png', os.path.join(DRIVE_OUTPUT, 'training_history.png'))
shutil.copy2('/content/confusion_matrix.png', os.path.join(DRIVE_OUTPUT, 'confusion_matrix.png'))

model_size = os.path.getsize(drive_model_path) / 1e6
print(f'✅ Modelo copiado a Drive: {drive_model_path}')
print(f'   Tamaño: {model_size:.2f} MB')
print(f'\n📥 SIGUIENTE PASO:')
print(f'   1. Descarga el archivo desde Drive: {drive_model_path}')
print(f'   2. En tu equipo ejecuta:')
print(f'      python scripts/download_from_colab.py <ruta_del_archivo_descargado>')

## Celda 12 — (Opcional) Descarga directa desde Colab
Si prefieres descargar el modelo directamente sin pasar por Drive.

In [ ]:
from google.colab import files
files.download(MODEL_PATH)
print('✅ Descarga iniciada. Revisa la carpeta de Descargas de tu navegador.')